# Health Data Analysis

This notebook explores a set of **1,014 patient health records** and looks at how
vital signs — blood pressure, body temperature and heart rate — relate to age and
to an overall risk level.

The goal is the path from a raw spreadsheet to clear, interactive charts that
highlight readings outside the healthy range: a bit of clean-up first, then
analysis and findings.

**Dataset:** `data/data_kesehatan.xlsx`

| Column | Meaning |
| --- | --- |
| `Age` | Patient age (years) |
| `SystolicBP` / `DiastolicBP` | Blood pressure (mmHg) |
| `BodyTemp(fahrenheit)` / `body Temp (celcius)` | Body temperature, stored twice |
| `HeartRate` | Heart rate (bpm) |
| `RiskLevel` | Labelled risk: low / mid / high |

## Setup

Reference ranges and thresholds reused throughout the analysis.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = "data/data_kesehatan.xlsx"

# Healthy reference ranges.
NORMAL_TEMP = (36.5, 37.5)   # degrees Celsius
NORMAL_HR = (60, 100)        # beats per minute
SYSTOLIC_MAX, DIASTOLIC_MAX = 120, 80  # upper bounds for normal blood pressure

RISK_ORDER = ["low risk", "mid risk", "high risk"]

## Data preparation

The dataset is mostly tidy but needs a little clean-up first:

- temperature is stored twice (Fahrenheit and Celsius) — keep Celsius only;
- the Celsius column name is awkward (`body Temp (celcius)`) — rename it;
- three temperature readings are blank — drop those rows.

After that, add the helper columns the charts rely on.

In [2]:
def load_data(path=DATA_PATH):
    df = pd.read_excel(path)
    df = df.drop(columns=["BodyTemp(fahrenheit)"])
    df = df.rename(columns={"body Temp (celcius)": "BodyTempC"})
    df = df.dropna(subset=["BodyTempC"])
    return df.sort_values("Age").reset_index(drop=True)


def bp_status(systolic, diastolic):
    if systolic < SYSTOLIC_MAX and diastolic < DIASTOLIC_MAX:
        return "normal"
    return "elevated"


df = load_data()
df["BPStatus"] = [bp_status(s, d) for s, d in zip(df.SystolicBP, df.DiastolicBP)]
df["TempStatus"] = df.BodyTempC.between(*NORMAL_TEMP).map({True: "normal", False: "abnormal"})
df["AgeBand"] = pd.cut(
    df.Age, [0, 20, 30, 40, 50, 100], labels=["<=20", "21-30", "31-40", "41-50", "50+"]
)

print(f"{len(df)} rows after cleaning")
df.head()

1011 rows after cleaning


,Age,SystolicBP,DiastolicBP,BodyTempC,HeartRate,RiskLevel,BPStatus,TempStatus,AgeBand
0,10,100,50,37.222222,70,mid risk,normal,normal,<=20
1,10,70,50,36.666667,70,low risk,normal,normal,<=20
2,10,85,65,36.666667,70,low risk,normal,normal,<=20
3,10,100,50,37.222222,70,mid risk,normal,normal,<=20
4,12,120,90,36.666667,80,mid risk,elevated,normal,<=20


## Dataset overview

Summary statistics for the numeric columns.

In [3]:
df[["Age", "SystolicBP", "DiastolicBP", "BodyTempC", "HeartRate"]].describe().round(1)

,Age,SystolicBP,DiastolicBP,BodyTempC,HeartRate
count,1011.0,1011.0,1011.0,1011.0,1011.0
mean,29.9,113.2,76.5,37.0,74.5
std,13.5,18.4,13.9,0.8,7.5
min,10.0,70.0,49.0,36.7,60.0
25%,19.0,100.0,65.0,36.7,70.0
50%,26.0,120.0,80.0,36.7,76.0
75%,39.0,120.0,90.0,36.7,80.0
max,70.0,160.0,100.0,39.4,90.0


## Risk level distribution

Most patients are labelled low risk, but a meaningful share fall into mid and high
risk — enough to make the comparisons below worthwhile.

In [4]:
counts = df.RiskLevel.value_counts().reindex(RISK_ORDER)
fig = px.bar(
    x=counts.index, y=counts.values, color=counts.index,
    color_discrete_map={"low risk": "#2ca02c", "mid risk": "#ff7f0e", "high risk": "#d62728"},
    labels={"x": "Risk level", "y": "Patients"},
    title="Patients by risk level",
)
fig.update_layout(showlegend=False)
fig.show()

## Blood pressure vs age

Blood pressure is the clearest story in the data. Plotting systolic readings against
age and marking the 120 mmHg line shows how readings drift above normal as patients
get older.

In [5]:
fig = px.scatter(
    df, x="Age", y="SystolicBP", color="BPStatus",
    color_discrete_map={"normal": "#2ca02c", "elevated": "#d62728"},
    opacity=0.6, labels={"SystolicBP": "Systolic BP (mmHg)"},
    title="Systolic blood pressure vs age",
)
fig.add_hline(y=SYSTOLIC_MAX, line_dash="dash", line_color="gray",
              annotation_text="120 mmHg")
fig.show()

### Share of elevated blood pressure by age band

Grouping the same flag by age band makes the trend hard to miss: elevated blood
pressure climbs steeply through middle age.

In [6]:
share = (
    df.assign(elevated=df.BPStatus.eq("elevated"))
      .groupby("AgeBand", observed=True)["elevated"].mean()
      .mul(100).round(1)
)
fig = px.bar(
    x=share.index, y=share.values,
    labels={"x": "Age band", "y": "Elevated BP (%)"},
    title="Share of patients with elevated blood pressure",
    text=share.values,
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_yaxes(range=[0, 110])
fig.show()

## Blood pressure by risk level

The risk labels line up with the vitals: average blood pressure rises step by step
from low to high risk, which is a good sanity check on the labelling.

In [7]:
means = df.groupby("RiskLevel")[["SystolicBP", "DiastolicBP"]].mean().reindex(RISK_ORDER).round(1)
fig = go.Figure()
fig.add_bar(name="Systolic", x=means.index, y=means.SystolicBP, marker_color="#1f77b4")
fig.add_bar(name="Diastolic", x=means.index, y=means.DiastolicBP, marker_color="#ff7f0e")
fig.update_layout(barmode="group", title="Average blood pressure by risk level",
                  yaxis_title="mmHg", xaxis_title="Risk level")
fig.show()

## Body temperature

Temperature is far more stable: most readings sit inside the 36.5–37.5 °C band, with
a small tail of higher readings.

In [8]:
fig = px.histogram(
    df, x="BodyTempC", color="TempStatus", nbins=20,
    color_discrete_map={"normal": "#2ca02c", "abnormal": "#d62728"},
    labels={"BodyTempC": "Body temperature (°C)"},
    title="Body temperature distribution",
)
fig.show()

## Heart rate

Every heart-rate reading in this dataset already falls within the healthy 60–100 bpm
range, so there is no abnormal group to flag here — the distribution is shown for
completeness.

In [9]:
fig = px.histogram(
    df, x="HeartRate", nbins=15,
    labels={"HeartRate": "Heart rate (bpm)"},
    title="Heart rate distribution",
)
fig.show()

## Key findings

- **Blood pressure is the dominant signal.** About **two thirds of patients** show
  elevated blood pressure (systolic ≥ 120 or diastolic ≥ 80 mmHg).
- **It rises sharply with age** — from roughly **41% at age 20 and under to nearly
  98% in the 41–50 band.**
- **The risk labels match the vitals:** high-risk patients average about **124/85
  mmHg** versus **106/73 mmHg** for low-risk ones, and they are older on average.
- **Temperature and heart rate are stable:** about **20%** of temperatures fall
  outside the normal band, while heart rate stays within range across the board.

In short, age and blood pressure are the strongest indicators of risk in this data,
while temperature and heart rate add little separation.